# Расширенная разметка сделок

Запускайте ноутбук из корня репозитория или из его папки. Список входных CSV задаётся в `INPUT_PATHS`. При объединении сохраняются все строки, включая повторяющиеся `listing_id`.

In [1]:
import os
import pandas as pd
from pathlib import Path
from IPython.display import display

repo_root = Path.cwd()
if repo_root.name in {"analysis", "feature", "models"}:
    repo_root = repo_root.parent

assert (repo_root / "requirements.txt").is_file() and (repo_root / "data/listings.csv").is_file(), (
    "Запустите ноутбук из корня репозитория или из папки, в которой он лежит"
)
os.chdir(repo_root)

In [2]:
from pathlib import Path

import pandas as pd

INPUT_PATHS = [
    Path("data/verified.csv"),
    Path("data_by_dewallen/survey_deal_labels.csv"),
    Path("analysis/verified_listing_of_messages.csv"),
]
OUTPUT_PATH = Path("analysis/extend_verified.csv")
COLUMNS = ["listing_id", "is_deal", "weight"]

In [3]:
label_tables = []
for path in INPUT_PATHS:
    labels = pd.read_csv(path, usecols=lambda column: column in COLUMNS)
    if "weight" not in labels.columns:
        labels["weight"] = 1
    label_tables.append(labels[COLUMNS])

extend_verified = pd.concat(label_tables, ignore_index=True)

In [4]:
# Приоритет NLP при совпадающих метках
extend_verified = (
    extend_verified
    .sort_values(
        ["is_deal", "weight"],
        ascending=[False, False]
    )
    .drop_duplicates("listing_id", keep="first")
)

# Загружаем эталонную разметку
original_verified = pd.read_csv("data/verified.csv")
original_verified["weight"] = 1.0

# Исключаем все эталонные ID из дополнительной разметки
extra = extend_verified[
    ~extend_verified["listing_id"].isin(
        original_verified["listing_id"]
    )
]

# Эталонная разметка всегда имеет приоритет
extend_verified = pd.concat(
    [original_verified, extra],
    ignore_index=True
)

assert extend_verified["listing_id"].is_unique

# Проверяем сохранность эталонной разметки
check = original_verified.merge(
    extend_verified,
    on="listing_id",
    suffixes=("_original", "_extended"),
    validate="one_to_one"
)

assert len(check) == len(original_verified)
assert (check["is_deal_original"] == check["is_deal_extended"]).all()
assert (check["weight_extended"] == 1.0).all()

print("Эталонная разметка сохранена без изменений")

Эталонная разметка сохранена без изменений


In [5]:
extend_verified.to_csv(OUTPUT_PATH, index=False)
print(f"Сохранено {len(extend_verified)} строк в {OUTPUT_PATH}")

Сохранено 3175 строк в analysis\extend_verified.csv


In [6]:
(extend_verified["weight"] == 0.8).sum()

np.int64(2149)

In [7]:
(extend_verified["weight"] == 1).sum()

np.int64(1026)

In [8]:
original = pd.read_csv("data/verified.csv")

extra = extend_verified[
    ~extend_verified["listing_id"].isin(
        original["listing_id"]
    )
]

print("Эталонных меток:", len(original))

print(
    "NLP-псевдометок:",
    extra["weight"].eq(1.0).sum()
)

display(extra.groupby(["is_deal", "weight"]).size())

Эталонных меток: 300
NLP-псевдометок: 726


is_deal  weight
1        0.8       2149
         1.0        726
dtype: int64